In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [14]:
print(os.listdir("../data"))

['.gitkeep', '10Y.csv', '3Y.csv', 'benchmark_map.csv', 'bond_return_proxies.csv', 'Brent_OIL.csv', 'CPI.csv', 'CPI_Sentiment.csv', 'credit_spreads.csv', 'DWA_Momentum_ETF.csv', 'etf_prices.csv', 'F-F_Momentum_Factor_daily.csv', 'index_prices.csv', 'ISM_Manufacturing.csv', 'ISM_Services.csv', 'M1.csv', 'M2.csv', 'NBER_US.csv', 'SPX.csv', 'treasury_yields.csv', 'VIX.csv']


<hr>
Equity Market Dynamics Features
<hr>

In [82]:
# ------------------------------------------------
# 1. Load index price data
# ------------------------------------------------
idx = pd.read_csv("../data/index_prices.csv", header=1)
idx.columns = ["Date"] + list(idx.columns[1:])

print("Columns in index_prices.csv:")
print(idx.columns.tolist())

spx = idx[["Date", "SPX Index"]].copy()
spx.dropna(how='any',inplace=True)

# Convert date and sort
spx["Date"] = pd.to_datetime(spx["Date"])
spx = spx.sort_values("Date").reset_index(drop=True)
spx["SPX Index"] = pd.to_numeric(spx["SPX Index"], errors="coerce")

# ------------------------------------------------
# 3. Compute features
# ------------------------------------------------

# Daily log return
spx["SPX_return"] = np.log(spx["SPX Index"] / spx["SPX Index"].shift(1))
#spx = spx

# 20-day rolling volatility of returns
spx["SPX_volatility_20d"] = spx["SPX_return"].dropna().rolling(20).std()

# 252-day rolling drawdown
rolling_max_252 = spx["SPX Index"].rolling(252, min_periods=1).max()
spx["SPX_drawdown"] = spx["SPX Index"] / rolling_max_252 - 1

# ------------------------------------------------
# Moving Average Features
# ------------------------------------------------

spx["SPX_MA50"] = spx["SPX Index"].rolling(50).mean()
spx["SPX_MA200"] = spx["SPX Index"].rolling(200).mean()

# continuous signal
spx["SPX_MA_signal"] = spx["SPX_MA50"] - spx["SPX_MA200"]


# ------------------------------------------------
# 4. Load Fama-French daily momentum factor
# ------------------------------------------------
mom = pd.read_csv(
    "../data/F-F_Momentum_Factor_daily.csv",
    header = 1,
    skiprows=13,
    skipfooter=1,
    names=["Date","Mom"],
    engine="python"
)


mom["Date"] = pd.to_datetime(mom["Date"], format="%Y%m%d")
mom["Mom"] = mom["Mom"] / 100


mom =  mom[mom.Date> pd.to_datetime('1999-12-31')]

spx = spx.merge(mom, on="Date", how="left")

spx =  spx[spx.Date< pd.to_datetime('2026-01-01')]


# ------------------------------------------------
# Select final Equity Market Dynamics features
# ------------------------------------------------

equity_market_dynamics_features = spx[[
    "Date",
    "SPX_return",
    "SPX_volatility_20d",
    "SPX_drawdown",
    "SPX_MA_signal",
    "Mom"
]].copy()

os.makedirs("../data/processed", exist_ok=True)

equity_market_dynamics_features.to_csv(
    "../data/processed/equity_market_dynamics_features.csv",
    index=False
)


Columns in index_prices.csv:
['Date', 'SPX Index', 'RLV Index', 'RU20INTR Index', 'RTY Index', 'LUACTRUU Index', 'LF98TRUU Index', 'GOLDLNPM Index']


<hr>
Volatility / Risk Sentiment 
<hr>

In [91]:



vix = pd.read_csv("../data/VIX.csv",header=0,names=["Date","VIX"])
vix["Date"] = pd.to_datetime(vix["Date"])
vix['VIXChange'] =  np.log(vix["VIX"] / vix["VIX"].shift(1))
vix['VIX3m-VIX'] = vix['VIX'] * (3**.5) - vix['VIX'] 
vix.dropna(how='any',inplace=True)

vix =  vix[vix.Date< pd.to_datetime('2026-01-01')]

Volatility_Risk_Sentiment_features = vix.copy()


Volatility_Risk_Sentiment_features.to_csv(
    "../data/processed/Volatility_Risk_Sentiment_features.csv",
    index=False
)

<hr>
Interest Rate Environment 
<hr>

In [110]:
yields = pd.read_csv("../data/treasury_yields.csv")
yields.columns = ["Date"] + list(yields.columns[1:])
yields["Date"] = pd.to_datetime(yields["Date"])

yields['Slope_10_2'] = yields['GT10 Govt'] - yields['GT2 Govt']
yields['Slope_30_10'] = yields['GT30 Govt'] - yields['GT10 Govt']

yields['YieldChange'] = yields['GT10 Govt'].shift(1) - yields['GT10 Govt']
yields['YieldCurvature'] = 2*yields['GT10 Govt'] - yields['GT2 Govt'] - yields['GT30 Govt']

yields.dropna(how='any',inplace=True)

Interest_Rate_Environment_features = yields.copy()

Interest_Rate_Environment_features.to_csv(
    "../data/processed/Interest_Rate_Environment_features.csv",
    index=False
)

<hr>
Cross-Asset Risk Signals 
<hr>

In [121]:
BondReturn = pd.read_csv("../data/bond_return_proxies.csv")
BondReturn.columns = ["Date"] + list(BondReturn.columns[1:])
BondReturn["Date"] = pd.to_datetime(BondReturn["Date"])


Bond_10y = BondReturn[['Date','IEF_proxy']]

Equity_Return = spx[['Date','SPX_return']]
Equity_Return = Equity_Return.dropna(how='any')


Equity_Bond = Equity_Return.merge(Bond_10y,on='Date',how='left')


Equity_Bond['Return(Eq-Bond)'] = Equity_Bond['SPX_return'] - Equity_Bond['IEF_proxy'] 
Equity_Bond["Equity_bond_corr"] = Equity_Bond["SPX_return"].rolling(60).corr(Equity_Bond['IEF_proxy'])

CrossAsset_RiskSignals_features = Equity_Bond.copy()

CrossAsset_RiskSignals_features.to_csv(
    "../data/processed/CrossAsset_RiskSignals_features.csv",
    index=False
)